# agentv24_production_deployment_pattern

This version wraps a LangGraph agent as a production-style FastAPI service.

API endpoints:

```text
GET  /health
GET  /ready
POST /invoke
POST /stream
```


## 1. Install dependencies

```bash
pip install -U fastapi uvicorn langgraph langchain langchain-openai langsmith python-dotenv httpx ipython jupyter
```


In [ ]:
# Optional: uncomment in a fresh environment.
# %pip install -U fastapi uvicorn langgraph langchain langchain-openai langsmith python-dotenv httpx ipython jupyter


## 2. Load environment variables


In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

cwd = Path.cwd()
load_dotenv(cwd / ".env")
load_dotenv(cwd.parent / ".env")

print("OpenAI API key present:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))


## 3. Define settings and state


In [ ]:
from dataclasses import dataclass
from typing import NotRequired, TypedDict

@dataclass(frozen=True)
class Settings:
    app_name: str = "agentv24-production-agent"
    model_name: str = os.getenv("MODEL_NAME", "gpt-4o-mini")
    temperature: float = float(os.getenv("TEMPERATURE", "0"))

settings = Settings()

class AgentState(TypedDict):
    input: str
    request_id: str
    context: NotRequired[dict]
    classification: NotRequired[str]
    analysis: NotRequired[str]
    final_answer: NotRequired[str]


## 4. Build graph


In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import START, END, StateGraph

def classify_request_node(state: AgentState) -> AgentState:
    text = state["input"].lower()

    if any(term in text for term in ["latency", "timeout", "failure", "incident", "error"]):
        classification = "incident_analysis"
    elif any(term in text for term in ["sql", "metric", "volume", "count"]):
        classification = "analytics"
    else:
        classification = "general"

    return {"classification": classification}

def analyze_node(state: AgentState) -> AgentState:
    llm = ChatOpenAI(model=settings.model_name, temperature=settings.temperature)

    prompt = f'''
You are a production LangGraph agent.

Request id:
{state["request_id"]}

Classification:
{state["classification"]}

Optional context:
{state.get("context", {})}

User request:
{state["input"]}

Produce a concise operational analysis.
'''

    response = llm.invoke(prompt)
    return {"analysis": response.content}

def final_answer_node(state: AgentState) -> AgentState:
    return {
        "final_answer": (
            f"Classification: {state['classification']}\n\n"
            f"{state['analysis']}"
        )
    }

def build_graph():
    graph_builder = StateGraph(AgentState)

    graph_builder.add_node("classify_request", classify_request_node)
    graph_builder.add_node("analyze", analyze_node)
    graph_builder.add_node("final_answer", final_answer_node)

    graph_builder.add_edge(START, "classify_request")
    graph_builder.add_edge("classify_request", "analyze")
    graph_builder.add_edge("analyze", "final_answer")
    graph_builder.add_edge("final_answer", END)

    return graph_builder.compile()

graph = build_graph()
graph


## 5. Test local graph invocation


In [ ]:
result = graph.invoke({
    "input": "Investigate CHECK-DOMAIN timeout spike after release R13.",
    "request_id": "req_notebook_001",
    "context": {
        "command": "CHECK-DOMAIN",
        "release": "R13",
    },
})

print(result["final_answer"])


## 6. Stream graph updates locally


In [ ]:
async for chunk in graph.astream(
    {
        "input": "Stream EPP incident analysis.",
        "request_id": "req_stream_notebook",
        "context": {"command": "CHECK-DOMAIN"},
    },
    stream_mode="updates",
):
    print("\n--- update ---")
    print(chunk)


## 7. Run API server

In terminal:

```bash
python -m agentv24_production_deployment_pattern.api_server
```

Then test:

```bash
python -m agentv24_production_deployment_pattern.smoke_client
```
